In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.segmentation import relabel_sequential

In [17]:
SC_data = pd.read_csv('/home/dean/Downloads/SingleCell_and_Metadata/Data_publication/ZurichTMA/SC_dat.csv')
#get unique values of 'core'
unique_cores = SC_data['core'].unique()
print(unique_cores)



['ZTMA208_slide_10_By10x4' 'ZTMA208_slide_10_By10x5'
 'ZTMA208_slide_10_By10x6' 'ZTMA208_slide_10_By10x7'
 'ZTMA208_slide_10_By10x8' 'ZTMA208_slide_11_By5x8'
 'ZTMA208_slide_11_By6x1' 'ZTMA208_slide_11_By6x2'
 'ZTMA208_slide_11_By6x3' 'ZTMA208_slide_11_By6x4'
 'ZTMA208_slide_12_Ay13x7' 'ZTMA208_slide_12_Ay13x8'
 'ZTMA208_slide_12_Ay14x1' 'ZTMA208_slide_12_Ay14x2'
 'ZTMA208_slide_12_Ay14x3' 'ZTMA208_slide_13_Cy9x1'
 'ZTMA208_slide_13_Cy9x2' 'ZTMA208_slide_13_Cy9x3'
 'ZTMA208_slide_13_Cy9x4' 'ZTMA208_slide_13_Cy9x5'
 'ZTMA208_slide_14_Cy3x4' 'ZTMA208_slide_14_Cy3x5'
 'ZTMA208_slide_14_Cy3x6' 'ZTMA208_slide_14_Cy3x7'
 'ZTMA208_slide_15_By13x6' 'ZTMA208_slide_15_By13x7'
 'ZTMA208_slide_15_By13x8' 'ZTMA208_slide_15_By14x1'
 'ZTMA208_slide_15_By14x2' 'ZTMA208_slide_16_Ay16x4'
 'ZTMA208_slide_16_Ay16x5' 'ZTMA208_slide_16_Ay16x6'
 'ZTMA208_slide_16_Ay16x7' 'ZTMA208_slide_16_Ay16x8'
 'ZTMA208_slide_16_By1x1' 'ZTMA208_slide_17_Ay8x1'
 'ZTMA208_slide_17_Ay8x2' 'ZTMA208_slide_17_Ay8x3'
 'ZTMA208_s

In [ ]:
unique_cores = SC_data['core'].unique()
print(unique_cores)

In [3]:
def load_label_mask(path):
    arr = imread(path)
    if arr.ndim > 2:
        arr = arr[..., 0]
    return arr.astype(np.int64)

def compute_intersections_and_areas(gt, pred):
    """Return intersection matrix, gt areas, pred areas."""
    gt, _, _ = relabel_sequential(gt)
    pred, _, _ = relabel_sequential(pred)

    max_gt = int(gt.max())
    max_pred = int(pred.max())

    # Flatten into joint label space
    combined = gt * (max_pred + 1) + pred
    hist = np.bincount(
        combined.ravel(),
        minlength=(max_gt + 1) * (max_pred + 1)
    )
    intersections = hist.reshape((max_gt + 1, max_pred + 1))

    area_gt = np.bincount(gt.ravel(), minlength=max_gt + 1)
    area_pred = np.bincount(pred.ravel(), minlength=max_pred + 1)

    return intersections, area_gt, area_pred, max_gt, max_pred

def compute_dice(gt, pred):
    """Foreground DICE over all pixels."""
    gt_fg = gt > 0
    pred_fg = pred > 0

    inter = np.logical_and(gt_fg, pred_fg).sum()
    size_gt = gt_fg.sum()
    size_pred = pred_fg.sum()

    denom = size_gt + size_pred
    if denom == 0:
        return np.nan
    return 2.0 * inter / denom

def compute_aji(intersections, area_gt, area_pred, max_gt, max_pred):
    """Aggregated Jaccard Index (AJI)."""
    gt_labels = np.arange(1, max_gt + 1)
    pred_labels = np.arange(1, max_pred + 1)

    matched_gt = set()
    matched_pred = set()
    inter_sum = 0
    union_sum = 0

    # Greedy matching: for each GT object, pick best overlapping pred
    for gi in gt_labels:
        overlaps = intersections[gi, :]
        overlaps[0] = 0  # ignore background
        if overlaps.max() == 0:
            continue

        # find best overlapping pred that is not yet matched
        candidate_idxs = np.argsort(overlaps)[::-1]  # descending
        pj = None
        for idx in candidate_idxs:
            if idx == 0:
                continue
            if idx not in matched_pred and overlaps[idx] > 0:
                pj = idx
                break

        if pj is None:
            continue

        inter = intersections[gi, pj]
        union = area_gt[gi] + area_pred[pj] - inter

        inter_sum += inter
        union_sum += union
        matched_gt.add(gi)
        matched_pred.add(pj)

    # Unmatched areas
    unmatched_gt = [i for i in gt_labels if i not in matched_gt]
    unmatched_pred = [j for j in pred_labels if j not in matched_pred]

    sum_unmatched_gt = area_gt[unmatched_gt].sum() if unmatched_gt else 0
    sum_unmatched_pred = area_pred[unmatched_pred].sum() if unmatched_pred else 0

    denom = union_sum + sum_unmatched_gt + sum_unmatched_pred
    if denom == 0:
        return np.nan
    return inter_sum / denom


def compute_over_under_seg(intersections, max_gt, max_pred):
    """
    Oversegmentation index (OI) and undersegmentation index (UI).

    OI: average excess predicted objects per GT object.
    UI: average excess GT objects per predicted object.
    """
    gt_labels = np.arange(1, max_gt + 1)
    pred_labels = np.arange(1, max_pred + 1)

    # Boolean overlap matrix excluding background
    overlaps_bool = intersections > 0
    overlaps_bool[0, :] = False
    overlaps_bool[:, 0] = False

    # Oversegmentation: GT -> multiple preds
    if len(gt_labels) > 0:
        over_values = []
        for gi in gt_labels:
            count = overlaps_bool[gi, 1:max_pred + 1].sum()
            over_values.append(max(count - 1, 0))
        oi = np.mean(over_values)
    else:
        oi = np.nan

    # Undersegmentation: Pred -> multiple GTs
    if len(pred_labels) > 0:
        under_values = []
        for pj in pred_labels:
            count = overlaps_bool[1:max_gt + 1, pj].sum()
            under_values.append(max(count - 1, 0))
        ui = np.mean(under_values)
    else:
        ui = np.nan

    return oi, ui


def compute_metrics_for_pair(ref_mask, pred_mask):
    """
    ref_mask: reference (Jackson) labels
    pred_mask: predicted labels (OpenIMC)
    """
    # Basic counts
    n_ref = np.unique(ref_mask)[np.unique(ref_mask) != 0].size
    n_pred = np.unique(pred_mask)[np.unique(pred_mask) != 0].size

    dice = compute_dice(ref_mask, pred_mask)

    intersections, area_gt, area_pred, max_gt, max_pred = compute_intersections_and_areas(
        ref_mask, pred_mask
    )
    aji = compute_aji(intersections, area_gt, area_pred, max_gt, max_pred)
    oi, ui = compute_over_under_seg(intersections, max_gt, max_pred)

    return {
        "dice": dice,
        "aji": aji,
        "overseg_index": oi,
        "underseg_index": ui,
        "n_ref": n_ref,
        "n_pred": n_pred,
        "obj_count_diff": n_pred - n_ref,
    }


In [3]:
def collect_common_files(ref_dir, pred_dir, exts=(".tif", ".tiff", ".png")):
    """Return sorted list of filenames present in both dirs with allowed extensions."""
    ref_files = {
        f for f in os.listdir(ref_dir)
        if os.path.isfile(os.path.join(ref_dir, f))
        and f.lower().endswith(exts)
    }
    pred_files = {
        f for f in os.listdir(pred_dir)
        if os.path.isfile(os.path.join(pred_dir, f))
        and f.lower().endswith(exts)
    }
    common = sorted(ref_files & pred_files)
    return common

In [4]:
reference_dir = '/home/dean/Downloads/OMEandSingleCellMasks/OMEnMasks/converted_masks/'
pred_dir = '/home/dean/Downloads/OMEandSingleCellMasks/OMEnMasks/cellsam_masks/'

In [5]:
common_files = collect_common_files(reference_dir, pred_dir)

In [6]:
records = []
for fname in common_files:
    ref_path = os.path.join(reference_dir, fname)
    pred_path = os.path.join(pred_dir, fname)
    ref_mask = load_label_mask(ref_path)
    pred_mask = load_label_mask(pred_path)
    metrics = compute_metrics_for_pair(ref_mask, pred_mask)
    metrics['filename'] = fname
    records.append(metrics)

df = pd.DataFrame(records)
df.to_csv('segmentation_benchmark_jackson.csv', index=False)


In [4]:
df = pd.read_csv('segmentation_benchmark_jackson.csv')

In [6]:
print("Summary statistics:")
print(df.describe())


Summary statistics:
             dice         aji  overseg_index  underseg_index        n_ref  \
count  735.000000  735.000000     735.000000      735.000000   735.000000   
mean     0.785776    0.306706       0.843598        1.469212  1730.291156   
std      0.131722    0.096944       0.423566        0.706525  1296.071019   
min      0.024977    0.001330       0.000000        0.038462     9.000000   
25%      0.766967    0.252179       0.548832        1.019704   678.000000   
50%      0.809842    0.317836       0.827383        1.395760  1563.000000   
75%      0.852347    0.367738       1.114908        1.823794  2455.000000   
max      0.963284    0.605010       2.137774        4.775484  6908.000000   

            n_pred  obj_count_diff  
count   735.000000      735.000000  
mean   1258.029932     -472.261224  
std     946.253098      495.051812  
min       3.000000    -5749.000000  
25%     482.000000     -630.500000  
50%    1083.000000     -401.000000  
75%    1849.000000     -169

In [9]:
#identify the fil with the lowest dice score
lowest_dice_file = df.loc[df['dice'].idxmin()]['filename']
print(f"File with lowest DICE score: {lowest_dice_file}")

#identify the file with the highest aji score
highest_aji_file = df.loc[df['aji'].idxmax()]['filename']


File with lowest DICE score: ZTMA208_slide_20.73kx15.16ky_7000x7000_6_20171115_150_48_By3x6_195_a0_full_ZTMA208_slide_20.73kx15.16ky_7000x7000_6_20171115_150_48_By3x6_195_a0_full_segmentation_masks.tif


In [12]:
def make_boxplots(df, output_prefix):
    metrics = ["dice", "aji", "overseg_index", "underseg_index", "obj_count_diff"]

    for metric in metrics:
        plt.figure()
        df.boxplot(column=metric)
        plt.title(metric)
        plt.ylabel(metric)
        plt.tight_layout()
        plt.savefig(f"{output_prefix}_{metric}_boxplot.png", dpi=300)
        plt.close()

    # Scatter: n_ref vs n_pred with fit line
    plt.figure()
    plt.scatter(df["n_ref"], df["n_pred"], alpha=0.7, label='Data')
    
    # Fit a line
    import numpy as np
    x = df["n_ref"].values
    y = df["n_pred"].values
    if len(x) > 1:
        m, b = np.polyfit(x, y, 1)
        fit_y = m * x + b
        # Calculate R^2
        ss_res = np.sum((y - fit_y) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else float('nan')
        plt.plot(x, fit_y, color='red', linestyle='--', label=f'Fit: y={m:.2f}x+{b:.2f}; $R^2$={r2:.3f}')
    
    plt.xlabel("Reference object count (Jackson)")
    plt.ylabel("Predicted object count (OpenIMC)")
    plt.title("Object counts per ROI")    
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_object_counts_scatter.png", dpi=300)
    plt.close()

In [13]:
make_boxplots(df, '.')

In [14]:
#identify the file with the biggest decrease in object count
biggest_decrease_file = df.loc[df['obj_count_diff'].idxmin()]['filename']
print(f"File with biggest decrease in object count: {biggest_decrease_file}")


File with biggest decrease in object count: ZTMA208_slide_13.25kx14.95ky_7000x7000_8_20171115_260_2_Cy1x4_62_a0_full_ZTMA208_slide_13.25kx14.95ky_7000x7000_8_20171115_260_2_Cy1x4_62_a0_full_segmentation_masks.tif
